In [17]:
!git clone https://github.com/crackerghost/reatime-voice-api.git
!git pull origin master

From https://github.com/crackerghost/reatime-voice-api
 * branch            master     -> FETCH_HEAD
Already up to date.


In [18]:
%cd /kaggle/working/reatime-voice-api

/kaggle/working/reatime-voice-api


In [19]:
!python -m pip install -q -r requirements.txt ctranslate2 faster-whisper


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.2/202.2 MB 8.9 MB/s eta 0:00:00:00:0100:01


In [20]:
import os
from kaggle_secrets import UserSecretsClient

# 1. API Keys
os.environ["GROQ_API_KEY"] = UserSecretsClient().get_secret("GROQ_API_KEY")

# 2. CUDA Memory & Stability (Prevents fragmentation without empty_cache lag)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["HF_HUB_DISABLE_XET"] = "1"

# 3. LLM (CRITICAL FOR LATENCY: Replaces slow reasoning model with ultra-fast Llama 3.3)
os.environ["LLM_MODEL"] = "openai/gpt-oss-20b"
os.environ["LLM_TEMPERATURE"] = "0.6"
os.environ["LLM_MAX_TOKENS"] = "600"

# 4. Turn-Taking & VAD (Silence cutoff for fast conversational flow)
os.environ["ASR_VAD_MODE"] = "server"             # Silero neural VAD (blocks fan/keyboard noise)
os.environ["VOICE_SILERO_SILENCE_MS"] = "550"     # 550ms silence = user finished speaking (was 1200ms!)
os.environ["VOICE_AUTO_SEND_MS"] = "550"
os.environ["VOICE_SPEAKER_GATE"] = "0"            # Disabled so students/users aren't rejected

# 5. ASR (faster-whisper on CUDA fp16)
os.environ["ASR_BACKEND"] = "faster-whisper"
os.environ["ASR_MODEL"] = "large-v3-turbo"        # Best Hindi/Hinglish accuracy (only ~1.6 GB VRAM)
os.environ["ASR_LANG"] = "hi"
os.environ["ASR_DEVICE"] = "cuda"
os.environ["ASR_COMPUTE"] = "float16"
os.environ["ASR_FINAL_BEAM"] = "1"                # Greedy is 4x faster (~80ms) and large-v3-turbo doesn't need beam 5
os.environ["ASR_INITIAL_PROMPT"] = "नमस्ते राहुल भाई, आप कैसे हैं? हाँ, मैं पूछ रहा हूँ कि मेरी आवाज़ आ रही है?"
# 6. TTS (OmniVoice on CUDA fp16)
os.environ["VOICE_API_DEVICE"] = "cuda"
os.environ["VOICE_API_DTYPE"] = "fp16"
os.environ["VOICE_NUM_STEP"] = "16"               # Subsequent windows get high quality (runs in background)
os.environ["VOICE_FIRST_STEP"] = "7"              # Sweet spot: 7 steps on T4 takes ~160ms and sounds clean
os.environ["VOICE_FIRST_WINDOW_CHARS"] = "26"     # ~4-6 words: enough text for natural intonation
os.environ["VOICE_MIN_WINDOW_CHARS"] = "24"
os.environ["VOICE_WINDOW_CHARS"] = "85"
os.environ["VOICE_TEMPERATURE"] = "0.3"
os.environ["VOICE_SPEED"] = "1.02"                # 2% faster gives a snappier conversational cadence

os.environ["VISION_BACKEND"] = "local"
os.environ["VISION_LOCAL_4BIT"] = "0"             # 4-bit is SLOWER on T4 (Turing dequant) — keep fp16
# 7. Voice Clone Reference
os.environ["VOICE_REF_TEXT"] = "कोडिंग में बहुत मज़ा आता है, बट समटाइम्स बग्स आर सो अनोइंग यार।"
os.environ["VOICE_REF_AUDIO"] = "/kaggle/input/datasets/rajsingha/bunty-updated/my_voice.wav"

print("🔥 Realtime Conversational Voice Pipeline Configured for Kaggle GPU!")

🔥 Realtime Conversational Voice Pipeline Configured for Kaggle GPU!


In [21]:
!node --version && cd web/ui && npm install --silent && npm run build

v20.19.0

> voice-ui@0.0.0 build
> vite build

vite v6.4.3 building for production...
transforming (1) src/main.jsxtransforming (7) src/index.csstransforming (18) src/AuraGlobe.jsxtransforming (23) node_modules/react-dom/index.js✓ 34 modules transformed.
rendering chunks (1)...computing gzip size (0)...computing gzip size (1)...computing gzip size (2)...computing gzip size (3)...dist/index.html                   0.44 kB │ gzip:  0.32 kB
dist/assets/index-CjFShVrb.css   26.08 kB │ gzip:  5.70 kB
dist/assets/index-BOb9B6GZ.js   184.12 kB │ gzip: 60.28 kB
✓ built in 1.74s
⠙

In [22]:
!pip -q install pyngrok
from pyngrok import ngrok
ngrok.set_auth_token("1b4EWhlNt3eQvoNL09GXdkkLUGS_5zPS1AU5cXQgY6xVu53D6")
print("PUBLIC URL:", ngrok.connect(8000))


PUBLIC URL: NgrokTunnel: "https://6287-35-237-182-25.ngrok-free.app" -> "http://localhost:8000"


t=2026-09-07T17:13:11+0000 lvl=warn msg="failed to open private leg" id=d969426d7cab privaddr=localhost:8000 err="dial tcp [::1]:8000: connect: connection refused"
t=2026-09-07T17:13:26+0000 lvl=warn msg="failed to open private leg" id=8542da087348 privaddr=localhost:8000 err="dial tcp [::1]:8000: connect: connection refused"


In [ ]:
!python voice_api.py


INFO:     Started server process [594]
INFO:     Waiting for application startup.
2026-09-07 17:13:49,899 Device: cuda | dtype: torch.float16
Fetching 13 files: 100%|█████████████████████| 13/13 [00:00<00:00, 11675.79it/s]
Download complete: : 0.00B [00:00, ?B/s]              
Loading weights: 100%|███████████████████████| 527/527 [00:01<00:00, 419.02it/s]
2026-09-07 17:13:57,414 OmniVoice loaded + voice prompt cached from my_voice.wav
2026-09-07 17:13:57,414 Loading ASR backend 'faster-whisper' (model 'large-v3-turbo')...
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
2026-09-07 17:13:57,589 HTTP Request: GET https://huggingface.co/api/models/mobiuslabsgmbh/faster-whisper-large-v3-turbo/revision/main "HTTP/1.1 307 Temporary Redirect"
2026-09-07 17:13:57,631 HTTP Request: GET https://huggingface.co/api/models/dropbox-dash/faster-whisper-large-v3-turbo/revision/main "HTTP/1.1 200 OK"
2026-09-07 17:13:59,679 ASR backend 'fa